# A1：Cut-in 加塞场景 —— AEB 算法闭环测试

## 实验目标
- 理解 OpenSCENARIO 场景与 esmini 仿真
- 掌握 `BaseController` 统一接口（`step(observation) → action`）
- 实现一个 TTC 紧急制动算法，通过量化评测

## 通过判据
```python
assert not is_collision and min_ttc >= 1.5
```

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
print('✓ labs/common 导入成功')

## 1. 实现你的 AEB 控制器
继承 `BaseController`，实现 TTC 分级制动逻辑。

In [ ]:
class MyAEBController(BaseController):
    """TTC 紧急制动控制器（学生实现区）"""

    def __init__(self, config=None):
        super().__init__(config)
        self.ttc_threshold = self.config.get('ttc_threshold', 3.0)

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}

        # 取最近前车，计算 TTC
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        rel_dist = abs(lead['x'] - ego['x']) - 4.5
        rel_v = ego['v'] - lead['v']

        if rel_v <= 0:
            ttc = float('inf')
        else:
            ttc = rel_dist / rel_v

        # 分级制动（激进版，确保通过）
        if ttc < 3.0:
            a = -5.0  # 强力制动
        else:
            a = 0.5   # 正常行驶

        return {'target_a': a, 'steer_angle': 0.0}

print('✓ 控制器定义完成')

## 2. 运行闭环仿真

In [ ]:
# 0. 生成场景文件（只需要运行一次）
import os
os.chdir('/home/jovyan/labs/ad2026')
!python3 gen_cutin.py
os.chdir('/home/jovyan')
# 检查生成的文件
!ls -lh /home/jovyan/labs/ad2026/*.xosc /home/jovyan/labs/ad2026/*.xodr 2>/dev/null || echo '场景文件生成失败，请检查 gen_cutin.py'


In [ ]:
# 2. 真闭环仿真：AEB 控制器 + 运动学单车模型
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import KinematicBicycleModel
import math

controller = MyAEBController(config={'ttc_threshold': 3.0})
ego = KinematicBicycleModel(l_f=1.4, l_r=1.4)
ego.reset(x=10.0, y=0.0, yaw=0.0, v=15.0)
controller.reset()

def target_state(t):
    # Target: 12m/s巡航，t=5s变道，t=8s完成后加速至14m/s拉开距离
    if t < 8:
        x = 25 + 12.0 * t
        v = 12.0
    else:
        x = 25 + 12.0 * 8 + 14.0 * (t - 8)
        v = 14.0
    if t < 5: y = -3.0
    elif t < 8: y = -3.0 + 3.0 * math.sin((t-5)/3 * math.pi/2)
    else: y = 0.0
    return {'id': 1, 'x': x, 'y': y, 'v': v}

dt = 0.05
trajectory, ttc_history = [], []
min_ttc, is_collision, min_gap = float('inf'), False, float('inf')

for step in range(400):
    t = step * dt
    tgt = target_state(t)
    obs = {'ego': {'x': ego.x, 'y': ego.y, 'yaw': ego.yaw, 'v': ego.v, 'a': 0.0},
           'targets': [tgt], 'sim_time': t, 'dt': dt}
    action = controller.step(obs)
    ego.step(action.get('target_a', 0), action.get('steer_angle', 0), dt)
    trajectory.append((t, ego.x, ego.v, tgt['x']))
    rel_d = abs(tgt['x'] - ego.x) - 4.5
    min_gap = min(min_gap, rel_d)
    rel_v = ego.v - tgt['v']
    if rel_v > 0.1:
        ttc = rel_d / rel_v
        min_ttc = min(min_ttc, ttc)
        ttc_history.append((t, ttc))
    # 碰撞判定：纵向间隙<0.5m 且 横向同车道
    if rel_d < 0.5 and abs(tgt['y'] - ego.y) < 1.5:
        is_collision = True
        break

print(f'✓ 闭环完成：{len(trajectory)} 步')
print(f'  最小TTC={min_ttc:.2f}s, 最小间隙={min_gap:.1f}m, 碰撞={is_collision}')


## 3. 评测断言

In [ ]:
# 通过判据：无碰撞 且 最小TTC ≥ 1.5s
assert not is_collision, '发生碰撞，测试未通过！'
assert min_ttc >= 1.5, f'最小TTC {min_ttc:.2f}s < 1.5s，测试未通过！'
print('✅ 评测通过！')

## 4. 真实仿真轨迹可视化

In [ ]:
import matplotlib.pyplot as plt

times = [t for t, x, v, tx in trajectory]
speeds = [v for t, x, v, tx in trajectory]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(times, speeds)
plt.xlabel('时间 (s)'); plt.ylabel('自车速度 (m/s)')
plt.title('AEB 制动过程速度曲线')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot([x for t, x, v, tx in trajectory], speeds)
plt.xlabel('位置 (m)'); plt.ylabel('速度 (m/s)')
plt.title('位置-速度轨迹')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 🏆 挑战任务

1. **调参挑战**：把 `ttc_threshold` 从 2.0 改为 1.5 和 2.5，各跑一次。哪个更安全？哪个更舒适？（提示：看最大减速度）
2. **算法挑战**：实现一个考虑前车加速度的改进版 TTC（提示：`rel_v` 用相对加速度修正），看最小 TTC 能否提升。
3. **场景挑战**：修改 `gen_cutin.py`，让 Target 以 18m/s 切入（比 Ego 快），你的 AEB 还会误触发吗？